# 1. Proje ve Problem Tanımı

Customer Churn (müşteri kaybı), bir müşterinin işletmenin ürün veya hizmetini kullanmayı bırakmasıdır. Mevcut bir müşteriyi kaybetmek gelir kaybına yol açabileceği ve yeni müşteri kazanmak ek maliyet oluşturabileceği için churn işletmeler açısından önemli bir konudur.

Bu projede müşterilerin demografik, hizmet, abonelik ve ödeme bilgileri incelenerek müşterinin ayrılıp ayrılmadığı anlaşılmaya çalışılacaktır. Bu, iki olası sonucu bulunan bir **sınıflandırma (classification)** problemidir. Hedef değişken `Churn` sütunudur: `Yes` müşterinin ayrıldığını, `No` ise ayrılmadığını ifade eder.

# 2. Veri Setinin Tanıtılması

IBM Telco Customer Churn veri setinde her satır bir müşteriyi temsil eder. Sütunlar; müşteri özelliklerini, kullanılan telefon ve internet hizmetlerini, abonelik ve ödeme bilgilerini ve müşterinin ayrılma durumunu içerir.

In [1]:
import pandas as pd

data_path = "data/Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print(f"Veri setinin boyutu: {df.shape[0]} satır × {df.shape[1]} sütun")
print("\nSütun isimleri:")
print(df.columns.tolist())

Veri setinin boyutu: 7043 satır × 21 sütun

Sütun isimleri:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [3]:
df.dtypes.rename("Veri Tipi").to_frame()

,Veri Tipi
customerID,str
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

`info()` çıktısına göre tüm sütunlarda 7.043 dolu kayıt görünmektedir. Ancak metin sütunlarında standart eksik değer olarak algılanmayan boş veya yalnızca boşluk içeren değerler ayrıca kontrol edilmelidir.

In [5]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.16,0.37,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.37,24.56,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.76,30.09,18.25,35.5,70.35,89.85,118.75


## Değişken Grupları

### Müşteri bilgileri

- `customerID`: Her müşteriye ait benzersiz kimlik.
- `gender`: Müşterinin cinsiyeti.
- `SeniorCitizen`: Müşterinin yaşlı vatandaş olup olmadığını gösteren 0/1 kodu.
- `Partner`: Müşterinin eşi veya partneri olup olmadığı.
- `Dependents`: Müşterinin bakmakla yükümlü olduğu kişiler olup olmadığı.

### Hizmet bilgileri

- `PhoneService` ve `MultipleLines`: Telefon hizmeti ve birden fazla hat kullanımı.
- `InternetService`: İnternet hizmetinin türü.
- `OnlineSecurity`, `OnlineBackup`, `DeviceProtection` ve `TechSupport`: İnternet güvenliği, yedekleme, cihaz koruma ve teknik destek hizmetleri.
- `StreamingTV` ve `StreamingMovies`: TV ve film yayın hizmetleri.

### Abonelik ve ödeme bilgileri

- `tenure`: Müşterinin şirkette kaldığı ay sayısı.
- `Contract`: Sözleşme türü.
- `PaperlessBilling`: Kağıtsız fatura kullanımı.
- `PaymentMethod`: Ödeme yöntemi.
- `MonthlyCharges` ve `TotalCharges`: Aylık ücret ve toplam ücret.

### Hedef

- `Churn`: Müşterinin ayrılıp ayrılmadığını gösteren hedef değişken (`Yes` / `No`).

# 3. Keşifsel Veri Analizi

Bu bölümde veri setinin yapısı ve temel veri kalitesi incelenmektedir. Tespit edilen sorunlar yalnızca not edilecek; bu aşamada veri temizleme veya kalıcı veri tipi dönüşümü yapılmayacaktır.

## Eksik ve Boş Değer Kontrolü

In [6]:
text_columns = df.select_dtypes(include=["object", "string"]).columns

missing_summary = pd.DataFrame({
    "Standart Eksik Değer (NaN)": df.isna().sum(),
    "Boş veya Yalnızca Boşluk": [
        df[column].astype("string").str.strip().eq("").sum()
        if column in text_columns else 0
        for column in df.columns
    ],
})

missing_summary

,Standart Eksik Değer (NaN),Boş veya Yalnızca Boşluk
customerID,0,0
gender,0,0
SeniorCitizen,0,0
Partner,0,0
Dependents,0,0
tenure,0,0
PhoneService,0,0
MultipleLines,0,0
InternetService,0,0
OnlineSecurity,0,0


Standart `NaN` kontrolünde eksik değer bulunmamıştır. Buna karşılık `TotalCharges` sütununda **11 adet yalnızca boşluk içeren değer** vardır. Bu değerler pandas tarafından standart eksik değer olarak sayılmamaktadır. Temizleme işlemi veri ön işleme aşamasına bırakılmıştır.

## Tekrarlanan Kayıt Kontrolü

In [7]:
duplicate_summary = pd.DataFrame({
    "Kontrol": ["Tekrarlanan tam satır", "Tekrarlanan customerID"],
    "Sayı": [df.duplicated().sum(), df["customerID"].duplicated().sum()],
})

duplicate_summary

,Kontrol,Sayı
0,Tekrarlanan tam satır,0
1,Tekrarlanan customerID,0


Veri setinde tekrarlanan tam satır veya tekrarlanan `customerID` bulunmamıştır.

## Kategorik Değişkenlerin Benzersiz Değerleri

`SeniorCitizen` sayısal saklansa da 0/1 ile iki grubu temsil ettiği için bu özette kategorik bir değişken olarak ele alınmıştır. Benzersiz müşteri kimliği olan `customerID` ve sayısal görünmesi beklenen `TotalCharges` bu listenin dışında tutulmuştur.

In [8]:
categorical_columns = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod", "Churn",
]

categorical_unique_summary = pd.DataFrame({
    "Sütun": categorical_columns,
    "Benzersiz Değer Sayısı": [df[column].nunique() for column in categorical_columns],
    "Değerler": [
        ", ".join(map(str, df[column].unique()))
        for column in categorical_columns
    ],
})

categorical_unique_summary

,Sütun,Benzersiz Değer Sayısı,Değerler
0,gender,2,"Female, Male"
1,SeniorCitizen,2,"0, 1"
2,Partner,2,"Yes, No"
3,Dependents,2,"No, Yes"
4,PhoneService,2,"No, Yes"
5,MultipleLines,3,"No phone service, No, Yes"
6,InternetService,3,"DSL, Fiber optic, No"
7,OnlineSecurity,3,"No, Yes, No internet service"
8,OnlineBackup,3,"Yes, No, No internet service"
9,DeviceProtection,3,"No, Yes, No internet service"


## Churn Dağılımı

In [9]:
churn_distribution = pd.DataFrame({
    "Adet": df["Churn"].value_counts(),
    "Yüzde (%)": df["Churn"].value_counts(normalize=True).mul(100).round(2),
})
churn_distribution.index.name = "Churn"

churn_distribution

,Adet,Yüzde (%)
Churn,,
No,5174,73.46
Yes,1869,26.54


Müşterilerin %73,46'sı ayrılmamış (`No`), %26,54'ü ayrılmıştır (`Yes`). `No` sınıfı belirgin biçimde daha büyüktür; dolayısıyla sınıflar tam dengeli değildir. Bu fark, ileride model değerlendirilirken yalnızca genel doğruluğa değil sınıf bazlı değerlendirme ölçülerine de bakmanın önemli olabileceğini gösterir. Bu aşamada herhangi bir modelleme kararı verilmemektedir.

## Seçili Kategorik Değişkenler

Aşağıdaki sayımlar veri setindeki bazı önemli kategorik değişkenlerin yapısını özetler.

In [10]:
selected_categorical_columns = [
    "Contract", "InternetService", "PaymentMethod", "gender", "SeniorCitizen"
]

selected_count_tables = []
for column in selected_categorical_columns:
    counts = df[column].value_counts()
    selected_count_tables.append(pd.DataFrame({
        "Değişken": column,
        "Değer": counts.index.astype(str),
        "Adet": counts.values,
        "Yüzde (%)": (counts.values / len(df) * 100).round(2),
    }))

selected_category_counts = pd.concat(selected_count_tables, ignore_index=True)
selected_category_counts

,Değişken,Değer,Adet,Yüzde (%)
0,Contract,Month-to-month,3875,55.02
1,Contract,Two year,1695,24.07
2,Contract,One year,1473,20.91
3,InternetService,Fiber optic,3096,43.96
4,InternetService,DSL,2421,34.37
5,InternetService,No,1526,21.67
6,PaymentMethod,Electronic check,2365,33.58
7,PaymentMethod,Mailed check,1612,22.89
8,PaymentMethod,Bank transfer (automatic),1544,21.92
9,PaymentMethod,Credit card (automatic),1522,21.61


## Sayısal Değişkenlerin Özeti

`tenure` ve `MonthlyCharges` sütunları doğrudan sayısal veri tipindedir. Temel istatistikleri aşağıda gösterilmiştir.

In [11]:
numeric_summary = (
    df[["tenure", "MonthlyCharges"]]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .T
    .round(2)
)

numeric_summary

,count,mean,std,min,median,max
tenure,7043.0,32.37,24.56,0.00,29.00,72.00
MonthlyCharges,7043.0,64.76,30.09,18.25,70.35,118.75


### TotalCharges Kontrolü

`TotalCharges` toplam ücret bilgisini içermesine rağmen metin (`str`) veri tipindedir. Yalnızca boşluk içeren 11 kayıt bu sütunun doğrudan sayısal olarak incelenmesini engeller. Aşağıdaki dönüşüm sadece problemi ölçmek için geçici bir seri üzerinde yapılır; özgün `df["TotalCharges"]` sütunu değiştirilmez. Kalıcı dönüşüm ve eksik değer işlemi veri ön işleme aşamasına bırakılmıştır.

In [12]:
total_charges_numeric_check = pd.to_numeric(df["TotalCharges"], errors="coerce")
total_charges_summary = (
    total_charges_numeric_check
    .agg(["count", "mean", "std", "min", "median", "max"])
    .rename("Değer")
    .to_frame()
    .round(2)
)

print(f"Özgün TotalCharges veri tipi: {df['TotalCharges'].dtype}")
print(f"Sayısala dönüşemeyen değer sayısı: {total_charges_numeric_check.isna().sum()}")
total_charges_summary

Özgün TotalCharges veri tipi: str
Sayısala dönüşemeyen değer sayısı: 11


,Değer
count,7032.00
mean,2283.30
std,2266.77
min,18.80
median,1397.48
max,8684.80


### İlk Bulgular

- Veri seti 7.043 müşteri ve 21 sütundan oluşmaktadır. Sütunlarda metin, tam sayı ve ondalıklı sayı veri tipleri bulunmaktadır.
- Standart eksik değer (`NaN`) yoktur; ancak `TotalCharges` sütununda standart kontrolde görünmeyen 11 adet yalnızca boşluk içeren değer vardır.
- Tekrarlanan tam satır ve tekrarlanan müşteri kimliği bulunmamıştır.
- `Churn` dağılımında `No` 5.174 (%73,46), `Yes` 1.869 (%26,54) kayıttır; sınıflar arasında belirgin bir sayı farkı vardır.
- `TotalCharges` metin tipindedir. Geçici sayısal kontrolde 7.032 değer analiz edilebilmiştir; kalıcı dönüşüm veri ön işleme aşamasına bırakılmıştır.

# 4. Veri Görselleştirme

# 5. Veri Ön İşleme

# 6. Modelin Oluşturulması

# 7. Modelin Değerlendirilmesi

# 8. Sonuç ve Yorumlar